# SpatialMind - LangSplat Training Pipeline

Complete training notebook for producing LangSplat artifacts on Google Colab A100.

**Output artifacts:**
- `point_cloud.ply` - Gaussians with XYZ, color, opacity, covariance
- `language_feature_dim3/*.npy` - Compressed 3-dim CLIP features per image
- `autoencoder.pth` - Trained 512-dim to 3-dim autoencoder

**Run top-to-bottom on a Colab A100 runtime.**

## Cell 0: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = "/content/drive/MyDrive/spatialMind"
os.makedirs(BASE, exist_ok=True)
os.makedirs(f"{BASE}/data", exist_ok=True)
os.makedirs(f"{BASE}/ckpts", exist_ok=True)
SCENE = "figurines"  # Change to "jachacks_venue" for live scan
DATASET_PATH = f"{BASE}/data/{SCENE}"
print(f"Base path: {BASE}")
print(f"Scene: {SCENE}")
print(f"Dataset path: {DATASET_PATH}")

## Cell 1: Environment Setup

Installs PyTorch with CUDA 11.8, clones LangSplat, compiles CUDA extensions, and installs all dependencies.

**Important:** Do NOT use LangSplat's `environment.yml` -- it pins Python 3.7 which is incompatible with Colab.

In [ ]:
# Force PyTorch with CUDA 11.8 (per research -- do NOT use env.yml, it pins Python 3.7)
!pip install torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118 \
  --index-url https://download.pytorch.org/whl/cu118

# Clone LangSplat with submodules (to Drive so it survives restarts)
import os
if not os.path.exists(f"{BASE}/LangSplat"):
    !git clone --recursive https://github.com/minghanqin/LangSplat.git {BASE}/LangSplat

# Compile and install CUDA extensions (against cu118)
!pip install {BASE}/LangSplat/submodules/langsplat-rasterization
!pip install {BASE}/LangSplat/submodules/simple-knn

# Install SAM (LangSplat fork -- NOT stock facebook/segment-anything)
!pip install git+https://github.com/minghanqin/segment-anything-langsplat.git

# Remaining dependencies
!pip install open-clip-torch plyfile tqdm tensorboard opencv-python-headless numpy

# COLMAP (CPU SIFT -- GPU SIFT not available via apt, but CPU is fine for 50-80 images)
!apt-get install -y colmap

# Verify installation
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
assert torch.cuda.is_available(), "CUDA not available! Check GPU runtime."
assert "cu118" in torch.__version__ or torch.version.cuda.startswith("11.8"), "CUDA 11.8 mismatch!"

## Cell 2: Download SAM Checkpoint + LERF Fallback Dataset

Downloads the SAM vit_h checkpoint (2.4 GB) and the LERF dataset (contains figurines, ramen, teatime, waldo_kitchen).
Both are saved to Google Drive to avoid re-downloading on session restart.

In [ ]:
import os

# SAM vit_h checkpoint (2.4 GB -- save to Drive to avoid re-download)
SAM_CKPT = f"{BASE}/ckpts/sam_vit_h_4b8939.pth"
if not os.path.exists(SAM_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -P {BASE}/ckpts/
    assert os.path.exists(SAM_CKPT), "SAM checkpoint download failed!"
    print(f"SAM checkpoint: {os.path.getsize(SAM_CKPT) / 1e9:.1f} GB")
else:
    print(f"SAM checkpoint already exists ({os.path.getsize(SAM_CKPT) / 1e9:.1f} GB)")

# LERF dataset (contains figurines, ramen, teatime, waldo_kitchen)
# Per D-04: lerf_figurines is the fallback scene
LERF_PATH = f"{BASE}/data/lerf_ovs"
if not os.path.exists(LERF_PATH):
    !pip install -q gdown
    !gdown 1QF1Po5p5DwTjFHu6tnTeYs_G0egMVmHt -O {BASE}/data/lerf_ovs.tar
    !tar -xf {BASE}/data/lerf_ovs.tar -C {BASE}/data/
    assert os.path.exists(LERF_PATH), "LERF dataset extraction failed!"
    print("LERF dataset extracted")
else:
    print("LERF dataset already exists")

# Verify figurines scene has COLMAP data
figurines_sparse = f"{BASE}/data/figurines/sparse/0/cameras.bin"
if os.path.exists(figurines_sparse):
    print("Figurines COLMAP data: OK (pre-computed in LERF dataset)")
else:
    print("WARNING: Figurines may need COLMAP processing")

## Cell 3: COLMAP SfM

Runs COLMAP Structure-from-Motion to extract camera poses from photos.
Skips automatically for LERF scenes that already have pre-computed `sparse/` data.

In [ ]:
%cd {BASE}/LangSplat

# Check if COLMAP data already exists (LERF dataset comes with sparse/)
sparse_path = f"{DATASET_PATH}/sparse/0/cameras.bin"
if os.path.exists(sparse_path):
    print(f"COLMAP data already exists at {sparse_path} -- skipping")
else:
    print(f"Running COLMAP on {DATASET_PATH}...")
    # LangSplat's convert.py wraps COLMAP's full pipeline
    !python convert.py -s {DATASET_PATH}
    assert os.path.exists(sparse_path), f"COLMAP failed! Check images in {DATASET_PATH}/input/"
    print("COLMAP complete")

## Cell 4: SAM + CLIP Preprocessing

Generates hierarchical SAM masks per image, then encodes each mask region with CLIP ViT-B/16.
Produces 512-dim CLIP features per mask stored as `.npy` files.

In [ ]:
%cd {BASE}/LangSplat

print(f"Running SAM + CLIP preprocessing on {DATASET_PATH}...")
!python preprocess.py --dataset_path {DATASET_PATH}

# Verify output
import glob
npy_files = glob.glob(f"{DATASET_PATH}/language_features/*_f.npy")
assert len(npy_files) > 0, f"Preprocessing failed! No .npy files in {DATASET_PATH}/language_features/"
print(f"Generated {len(npy_files)} language feature files")

## Cell 5: Autoencoder Training (512-dim -> 3-dim Compression)

Trains an autoencoder to compress 512-dim CLIP features to 3-dim for efficient Gaussian storage.
Then generates compressed 3-dim features for all images.

In [ ]:
%cd {BASE}/LangSplat/autoencoder

print("Training autoencoder (512-dim -> 3-dim)...")
!python train.py \
  --dataset_path {DATASET_PATH} \
  --encoder_dims 256 128 64 32 3 \
  --decoder_dims 16 32 64 128 256 256 512 \
  --lr 0.0007 \
  --output {DATASET_PATH}/ae_ckpt

# Generate compressed 3-dim features
print("Generating compressed 3-dim features...")
!python test.py --dataset_path {DATASET_PATH} --output {DATASET_PATH}

# Verify
import glob
dim3_files = glob.glob(f"{DATASET_PATH}/language_feature_dim3/*_f.npy")
assert len(dim3_files) > 0, "Autoencoder test.py failed! No dim3 feature files."
print(f"Generated {len(dim3_files)} compressed feature files")

## Cell 6: Base 3DGS RGB Training (30K Iterations)

Trains the base 3D Gaussian Splatting model for RGB reconstruction.
Checkpoints saved every 5K iterations to Google Drive for crash recovery.
Automatically resumes from the last checkpoint if one exists.

In [ ]:
%cd {BASE}/LangSplat

OUTPUT_DIR = f"{DATASET_PATH}/output/{SCENE}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Check for existing checkpoint to resume from (per D-07)
import glob
existing_ckpts = sorted(glob.glob(f"{OUTPUT_DIR}/chkpnt*.pth"))
if existing_ckpts:
    last_ckpt = existing_ckpts[-1]
    print(f"Resuming from checkpoint: {last_ckpt}")
    !python train.py -s {DATASET_PATH} \
      -m {OUTPUT_DIR} \
      --start_checkpoint {last_ckpt} \
      --iterations 30000 \
      --save_iterations 5000 10000 15000 20000 25000 30000
else:
    print("Starting fresh 3DGS RGB training...")
    !python train.py -s {DATASET_PATH} \
      -m {OUTPUT_DIR} \
      --iterations 30000 \
      --save_iterations 5000 10000 15000 20000 25000 30000

# Verify checkpoint
ckpt_path = f"{OUTPUT_DIR}/chkpnt30000.pth"
assert os.path.exists(ckpt_path), f"3DGS training incomplete! Missing {ckpt_path}"
print(f"3DGS RGB training complete. Checkpoint: {os.path.getsize(ckpt_path) / 1e6:.1f} MB")

## Cell 7: LangSplat Training (3 Feature Levels)

Trains LangSplat on top of the RGB checkpoint for all three hierarchical feature levels (0, 1, 2).
Each level corresponds to a different SAM mask granularity.

In [ ]:
%cd {BASE}/LangSplat

OUTPUT_DIR = f"{DATASET_PATH}/output/{SCENE}"
ckpt_path = f"{OUTPUT_DIR}/chkpnt30000.pth"
assert os.path.exists(ckpt_path), f"RGB checkpoint missing at {ckpt_path}! Run Cell 6 first."

for level in [0, 1, 2]:
    level_output = f"{OUTPUT_DIR}_{level}"
    print(f"\n{'='*50}")
    print(f"Training LangSplat feature level {level}...")
    print(f"Output: {level_output}")
    print(f"{'='*50}")
    !python train.py -s {DATASET_PATH} \
      -m {level_output} \
      --start_checkpoint {ckpt_path} \
      --feature_level {level}

print("\nLangSplat training complete for all 3 feature levels!")

## Cell 8: Smoke Test (Cosine Similarity Verification)

Verifies that trained embeddings are semantically meaningful by computing cosine similarity
between decoded CLIP features and text queries. Real object queries should show higher variance
than random gibberish.

In [ ]:
import numpy as np
import torch
import open_clip
import glob

OUTPUT_DIR = f"{DATASET_PATH}/output/{SCENE}"

# Load autoencoder
ae_path = f"{DATASET_PATH}/ae_ckpt/best_ckpt.pth"
if not os.path.exists(ae_path):
    ae_path = f"{DATASET_PATH}/ae_ckpt/last_ckpt.pth"
assert os.path.exists(ae_path), f"Autoencoder not found! Checked best_ckpt.pth and last_ckpt.pth in {DATASET_PATH}/ae_ckpt/"

autoencoder = torch.load(ae_path)
autoencoder.eval()

# Load sample compressed features
dim3_files = sorted(glob.glob(f"{DATASET_PATH}/language_feature_dim3/*_f.npy"))
assert len(dim3_files) > 0, "No dim3 feature files found!"

features_3d = np.load(dim3_files[0])  # shape: (N_masks, 3)
features_3d_tensor = torch.tensor(features_3d).cuda().float()

# Decode to 512-dim
with torch.no_grad():
    features_512 = autoencoder.decode(features_3d_tensor)
    features_512 = features_512 / features_512.norm(dim=-1, keepdim=True)

# Encode text queries with CLIP ViT-B-16 (same model used in training)
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', pretrained='laion2b_s34b_b88k')
tokenizer = open_clip.get_tokenizer('ViT-B-16')
model.eval().cuda()

texts = ["chair", "table", "wall", "floor", "random gibberish xyz"]
text_tokens = tokenizer(texts).cuda()
with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

# Compute and display cosine similarity
print("="*60)
print("SMOKE TEST: Cosine Similarity")
print("="*60)
pass_count = 0
for i, text in enumerate(texts):
    sims = (features_512 @ text_features[i]).cpu().numpy()
    std = sims.std()
    print(f"  '{text}': min={sims.min():.3f}, max={sims.max():.3f}, mean={sims.mean():.3f}, std={std:.3f}")
    if text != "random gibberish xyz" and std > 0.05:
        pass_count += 1

print(f"\nPASS criteria: std > 0.05 for real object queries")
print(f"Result: {pass_count}/4 real queries passed")

if pass_count >= 2:
    print("\n*** SMOKE TEST PASSED -- embeddings are semantically meaningful ***")
else:
    print("\n*** SMOKE TEST FAILED -- embeddings may be noise. Check training. ***")

## Cell 9: Consolidate Artifacts to Single Directory

Copies all three artifact types (PLY, autoencoder, dim3 features) into a single
`artifacts/` directory for easy downstream consumption. This directory is the canonical
location for all downstream phases.

In [ ]:
import os
import shutil
import glob

OUTPUT_DIR = f"{DATASET_PATH}/output/{SCENE}"
ARTIFACTS_DIR = f"{DATASET_PATH}/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Consolidating artifacts to: {ARTIFACTS_DIR}")
print("="*60)

# 1. Copy point_cloud.ply
ply_src = f"{OUTPUT_DIR}/point_cloud/iteration_30000/point_cloud.ply"
ply_dst = f"{ARTIFACTS_DIR}/point_cloud.ply"
if os.path.exists(ply_src):
    shutil.copy2(ply_src, ply_dst)
    print(f"  PLY: {os.path.getsize(ply_dst) / 1e6:.1f} MB -> {ply_dst}")
else:
    print(f"  PLY: MISSING at {ply_src}")

# 2. Copy autoencoder checkpoint
ae_src = f"{DATASET_PATH}/ae_ckpt/best_ckpt.pth"
if not os.path.exists(ae_src):
    ae_src = f"{DATASET_PATH}/ae_ckpt/last_ckpt.pth"
ae_dst = f"{ARTIFACTS_DIR}/autoencoder.pth"
if os.path.exists(ae_src):
    shutil.copy2(ae_src, ae_dst)
    print(f"  Autoencoder: {os.path.getsize(ae_dst) / 1e6:.1f} MB -> {ae_dst}")
else:
    print(f"  Autoencoder: MISSING (checked best_ckpt.pth and last_ckpt.pth in ae_ckpt/)")

# 3. Copy language_feature_dim3 .npy files
dim3_src_dir = f"{DATASET_PATH}/language_feature_dim3"
dim3_dst_dir = f"{ARTIFACTS_DIR}/language_feature_dim3"
os.makedirs(dim3_dst_dir, exist_ok=True)
npy_files = glob.glob(f"{dim3_src_dir}/*_f.npy")
if npy_files:
    for npy_src in npy_files:
        npy_dst = f"{dim3_dst_dir}/{os.path.basename(npy_src)}"
        shutil.copy2(npy_src, npy_dst)
    total_npy_mb = sum(os.path.getsize(f"{dim3_dst_dir}/{os.path.basename(f)}") for f in npy_files) / 1e6
    print(f"  Dim3 features: {len(npy_files)} files, {total_npy_mb:.1f} MB total -> {dim3_dst_dir}/")
else:
    print(f"  Dim3 features: MISSING at {dim3_src_dir}")

# Summary
print("\n" + "="*60)
print("ARTIFACT CONSOLIDATION SUMMARY")
print("="*60)
all_ok = True
for name, path in [("point_cloud.ply", ply_dst), ("autoencoder.pth", ae_dst)]:
    exists = os.path.exists(path)
    all_ok = all_ok and exists
    status = f"{os.path.getsize(path) / 1e6:.1f} MB" if exists else "MISSING"
    print(f"  {name}: {status}")

dim3_count = len(glob.glob(f"{dim3_dst_dir}/*_f.npy"))
all_ok = all_ok and dim3_count > 0
print(f"  language_feature_dim3/: {dim3_count} files")

if all_ok:
    print(f"\n*** ALL ARTIFACTS CO-LOCATED at {ARTIFACTS_DIR} ***")
    print("Downstream phases should load from this directory.")
else:
    print("\n*** SOME ARTIFACTS MISSING -- check training outputs above ***")